In [3]:
import duckdb
import pandas as pd
from pathlib import Path

# Fix: go up one level from notebooks/ to project root
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

con = duckdb.connect(str(PROJECT_ROOT / "data" / "f1.db"))

# Verify paths are correct
print(f"Project root: {PROJECT_ROOT}")
print(f"Data raw: {DATA_RAW}")
print(f"CSVs found: {list(DATA_RAW.glob('*.csv'))}")

# Load all 8 CSVs
tables = [
    "f1_circuits",
    "f1_drivers", 
    "f1_constructors",
    "f1_races",
    "f1_results",
    "f1_qualifying",
    "f1_driver_standings",
    "f1_constructor_standings"
]

for table in tables:
    con.execute(f"""
        CREATE OR REPLACE TABLE {table} AS 
        SELECT * FROM read_csv_auto('{DATA_RAW}/{table}.csv')
    """)
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<35} {count:>6} rows")

Project root: /home/humza/projects/f1-rag
Data raw: /home/humza/projects/f1-rag/data/raw
CSVs found: [PosixPath('/home/humza/projects/f1-rag/data/raw/f1_constructors.csv'), PosixPath('/home/humza/projects/f1-rag/data/raw/f1_races.csv'), PosixPath('/home/humza/projects/f1-rag/data/raw/f1_constructor_standings.csv'), PosixPath('/home/humza/projects/f1-rag/data/raw/f1_qualifying.csv'), PosixPath('/home/humza/projects/f1-rag/data/raw/f1_results.csv'), PosixPath('/home/humza/projects/f1-rag/data/raw/f1_driver_standings.csv'), PosixPath('/home/humza/projects/f1-rag/data/raw/f1_drivers.csv'), PosixPath('/home/humza/projects/f1-rag/data/raw/f1_circuits.csv')]
f1_circuits                             78 rows
f1_drivers                             879 rows
f1_constructors                        214 rows
f1_races                              1171 rows
f1_results                           25939 rows
f1_qualifying                        11036 rows
f1_driver_standings                   3128 rows
f1_c

In [4]:
# Understand what columns you're working with
for table in tables:
    print(f"\n{'='*50}")
    print(f"{table}")
    print('='*50)
    con.execute(f"DESCRIBE {table}").df()


f1_circuits

f1_drivers

f1_constructors

f1_races

f1_results

f1_qualifying

f1_driver_standings

f1_constructor_standings


In [6]:
# Inspect actual column names for the key tables
for table in ["f1_results", "f1_races", "f1_drivers", "f1_constructors"]:
    print(f"\n{table}:")
    print(con.execute(f"DESCRIBE {table}").df()["column_name"].tolist())


f1_results:
['season', 'round', 'race_name', 'driver_id', 'driver_name', 'constructor_id', 'constructor', 'grid', 'position', 'points', 'laps', 'status', 'time', 'fastest_lap', 'fastest_lap_rank']

f1_races:
['season', 'round', 'race_name', 'circuit_id', 'date', 'time', 'url']

f1_drivers:
['driver_id', 'given_name', 'family_name', 'dob', 'nationality', 'number', 'code', 'url']

f1_constructors:
['constructor_id', 'name', 'nationality', 'url']


In [7]:
for table in ["f1_circuits", "f1_qualifying", "f1_driver_standings", "f1_constructor_standings"]:
    print(f"\n{table}:")
    print(con.execute(f"DESCRIBE {table}").df()["column_name"].tolist())


f1_circuits:
['circuit_id', 'circuit_name', 'locality', 'country', 'lat', 'lng', 'url']

f1_qualifying:
['season', 'round', 'driver_id', 'constructor_id', 'position', 'q1', 'q2', 'q3']

f1_driver_standings:
['season', 'position', 'points', 'wins', 'driver_id', 'constructor']

f1_constructor_standings:
['season', 'position', 'points', 'wins', 'constructor_id', 'constructor']


In [8]:
# Check what position values actually look like
print(con.execute("SELECT DISTINCT position FROM f1_results ORDER BY position LIMIT 20").df())

    position
0          1
1          2
2          3
3          4
4          5
5          6
6          7
7          8
8          9
9         10
10        11
11        12
12        13
13        14
14        15
15        16
16        17
17        18
18        19
19        20


In [9]:
# Q1: Most race wins of all time
con.execute("""
    SELECT driver_name, COUNT(*) AS wins
    FROM f1_results
    WHERE CAST(position AS INTEGER) = 1
    GROUP BY driver_name
    ORDER BY wins DESC
    LIMIT 10
""").df()

,driver_name,wins
0,Lewis Hamilton,105
1,Michael Schumacher,91
2,Max Verstappen,71
3,Sebastian Vettel,53
4,Alain Prost,51
5,Ayrton Senna,41
6,Fernando Alonso,32
7,Nigel Mansell,31
8,Jackie Stewart,27
9,Niki Lauda,25


In [10]:
# Q2: Constructor dominance by season
con.execute("""
    SELECT constructor, season, COUNT(*) AS wins
    FROM f1_results
    WHERE CAST(position AS INTEGER) = 1
    GROUP BY constructor, season
    ORDER BY season DESC, wins DESC
    LIMIT 20
""").df()

,constructor,season,wins
0,Mercedes,2026,3
1,McLaren,2025,14
2,Red Bull,2025,8
3,Mercedes,2025,2
4,Red Bull,2024,9
5,McLaren,2024,6
6,Ferrari,2024,5
7,Mercedes,2024,4
8,Red Bull,2023,21
9,Ferrari,2023,1


In [11]:
# Q3: Most podiums at a single circuit
con.execute("""
    SELECT res.driver_name, ci.circuit_name, COUNT(*) AS podiums
    FROM f1_results res
    JOIN f1_races r ON res.season = r.season AND res.round = r.round
    JOIN f1_circuits ci ON r.circuit_id = ci.circuit_id
    WHERE CAST(res.position AS INTEGER) <= 3
    GROUP BY res.driver_name, ci.circuit_name
    ORDER BY podiums DESC
    LIMIT 15
""").df()

,driver_name,circuit_name,podiums
0,Lewis Hamilton,Silverstone Circuit,15
1,Lewis Hamilton,Hungaroring,12
2,Michael Schumacher,Autodromo Enzo e Dino Ferrari,12
3,Lewis Hamilton,Circuit de Barcelona-Catalunya,12
4,Michael Schumacher,Circuit Gilles Villeneuve,12
5,Michael Schumacher,Circuit de Barcelona-Catalunya,12
6,Lewis Hamilton,Circuit de Spa-Francorchamps,11
7,Michael Schumacher,Circuit de Nevers Magny-Cours,11
8,Lewis Hamilton,Bahrain International Circuit,11
9,Michael Schumacher,Autódromo José Carlos Pace,10


In [12]:
# Q4: Does pole position predict the win?
con.execute("""
    SELECT grid AS start_position,
           COUNT(*) AS total_races,
           SUM(CASE WHEN CAST(position AS INTEGER) = 1 THEN 1 ELSE 0 END) AS wins,
           ROUND(100.0 * SUM(CASE WHEN CAST(position AS INTEGER) = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_pct
    FROM f1_results
    WHERE grid > 0
    GROUP BY grid
    ORDER BY grid
    LIMIT 10
""").df()

,start_position,total_races,wins,win_pct
0,1,1163,500.0,43.0
1,2,1153,272.0,23.6
2,3,1157,140.0,12.1
3,4,1159,69.0,6.0
4,5,1159,49.0,4.2
5,6,1152,40.0,3.5
6,7,1162,23.0,2.0
7,8,1156,17.0,1.5
8,9,1159,5.0,0.4
9,10,1157,12.0,1.0


In [13]:
# Q5: Closest championship battles
con.execute("""
    SELECT season,
           MAX(points) AS winner_points,
           MAX(points) - MIN(points) FILTER (WHERE CAST(position AS INTEGER) = 2) AS margin
    FROM f1_driver_standings
    GROUP BY season
    ORDER BY margin ASC
    LIMIT 10
""").df()

,season,winner_points,margin
0,1984,72.0,0.5
1,1958,42.0,1.0
2,2007,110.0,1.0
3,1981,50.0,1.0
4,1994,92.0,1.0
5,1964,40.0,1.0
6,2008,98.0,1.0
7,1961,34.0,1.0
8,1986,72.0,2.0
9,1999,76.0,2.0


In [14]:
# Q6: DNF rate by constructor (reliability)
con.execute("""
    SELECT constructor,
           COUNT(*) AS total_entries,
           SUM(CASE WHEN status != 'Finished' 
                    AND status NOT LIKE '+%' THEN 1 ELSE 0 END) AS dnfs,
           ROUND(100.0 * SUM(CASE WHEN status != 'Finished' 
                    AND status NOT LIKE '+%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS dnf_pct
    FROM f1_results
    GROUP BY constructor
    HAVING total_entries > 200
    ORDER BY dnf_pct ASC
    LIMIT 10
""").df()

,constructor,total_entries,dnfs,dnf_pct
0,Mercedes,705,83.0,11.8
1,Red Bull,842,155.0,18.4
2,Force India,424,88.0,20.8
3,Toro Rosso,536,147.0,27.4
4,Toyota,280,81.0,28.9
5,Ferrari,2486,739.0,29.7
6,Haas F1 Team,434,130.0,30.0
7,McLaren,1957,591.0,30.2
8,Aston Martin,244,74.0,30.3
9,Williams,1719,552.0,32.1


In [15]:
# Q7: Qualifying vs race — who gains or loses most places
con.execute("""
    SELECT res.driver_name,
           COUNT(*) AS races,
           ROUND(AVG(CAST(res.position AS INTEGER) - CAST(q.position AS INTEGER)), 2) AS avg_places_lost
    FROM f1_results res
    JOIN f1_qualifying q 
        ON res.season = q.season 
        AND res.round = q.round 
        AND res.driver_id = q.driver_id
    WHERE q.position IS NOT NULL 
      AND TRY_CAST(res.position AS INTEGER) IS NOT NULL
    GROUP BY res.driver_name
    HAVING COUNT(*) > 30
    ORDER BY avg_places_lost DESC
    LIMIT 10
""").df()

,driver_name,races,avg_places_lost
0,Gerhard Berger,46,4.15
1,Mika Häkkinen,62,3.89
2,Eddie Irvine,64,3.17
3,Damon Hill,59,2.93
4,Heinz-Harald Frentzen,81,2.58
5,Jean Alesi,62,2.55
6,Mark Webber,202,2.13
7,Ralf Schumacher,107,2.07
8,Juan Pablo Montoya,64,2.05
9,Michael Schumacher,192,1.62


In [16]:
# Q8: Which circuits produce most overtakes
con.execute("""
    SELECT ci.circuit_name,
           ROUND(AVG(ABS(CAST(res.position AS INTEGER) - res.grid)), 2) AS avg_position_change
    FROM f1_results res
    JOIN f1_races r ON res.season = r.season AND res.round = r.round
    JOIN f1_circuits ci ON r.circuit_id = ci.circuit_id
    WHERE res.grid > 0 
      AND TRY_CAST(res.position AS INTEGER) IS NOT NULL
    GROUP BY ci.circuit_name
    HAVING COUNT(*) > 50
    ORDER BY avg_position_change DESC
    LIMIT 10
""").df()

,circuit_name,avg_position_change
0,Indianapolis Motor Speedway,8.28
1,Phoenix street circuit,8.03
2,Nivelles-Baulers,7.75
3,Zolder,7.42
4,Autódromo Internacional Nelson Piquet,7.24
5,Detroit Street Circuit,7.04
6,Las Vegas Street Circuit,6.94
7,Kyalami,6.92
8,Adelaide Street Circuit,6.78
9,Scandinavian Raceway,6.70


In [17]:
# Q9: Points per decade by constructor
con.execute("""
    SELECT (season // 10 * 10) AS decade,
           constructor,
           SUM(points) AS total_points
    FROM f1_results
    GROUP BY decade, constructor
    ORDER BY decade DESC, total_points DESC
    LIMIT 20
""").df()

,decade,constructor,total_points
0,2020,Red Bull,3374.5
1,2020,Mercedes,3026.5
2,2020,Ferrari,2367.5
3,2020,McLaren,2314.0
4,2020,Aston Martin,572.0
5,2020,Alpine F1 Team,534.0
6,2020,AlphaTauri,306.0
7,2020,Racing Point,210.0
8,2020,Williams,200.0
9,2020,Haas F1 Team,186.0


In [18]:
# Q10: Driver career longevity
con.execute("""
    SELECT driver_name,
           MIN(season) AS first_season,
           MAX(season) AS last_season,
           MAX(season) - MIN(season) AS career_years,
           COUNT(DISTINCT season) AS active_seasons
    FROM f1_results
    GROUP BY driver_name
    ORDER BY career_years DESC
    LIMIT 15
""").df()

,driver_name,first_season,last_season,career_years,active_seasons
0,Fernando Alonso,2001,2026,25,23
1,Michael Schumacher,1991,2012,21,19
2,Kimi Räikkönen,2001,2021,20,19
3,Lewis Hamilton,2007,2026,19,20
4,Rubens Barrichello,1993,2011,18,19
5,Graham Hill,1958,1975,17,18
6,Jenson Button,2000,2017,17,18
7,Luca Badoer,1993,2009,16,5
8,Nico Hülkenberg,2010,2026,16,15
9,Riccardo Patrese,1977,1993,16,17


In [20]:
print("Notebook 1 complete.")
print("Schema understood, foreign keys mapped, 10 analytical queries working.")

Notebook 1 complete.
Schema understood, foreign keys mapped, 10 analytical queries working.
